BẢN ĐỒ THANH GHI TRONG MCU
                                             │
         ┌───────────────────────────────────┴───────────────────────────────────┐
         ▼                                                                       ▼
┌─────────────────────────────────┐                             ┌─────────────────────────────────┐
│     CPU CORE REGISTERS          │                             │  PERIPHERAL REGISTERS (SFRs)    │
│  (Nằm bên trong Lõi CPU)        │                             │ (Nằm ở các Khối Ngoại vi)       │
├─────────────────────────────────┤                             ├─────────────────────────────────┤
│ • General Purpose: R0 - R12     │                             │ • Control Registers (CR)        │
│ • Stack Pointer (SP)            │                             │ • Status Registers (SR)         │
│ • Link Register (LR)            │                             │ • Data Registers (RDR / TDR)    │
│ • Program Counter (PC)          │                             │ • Shift Registers (Phần cứng)   │
│ • Program Status Register (xPSR)│                             └─────────────────────────────────┘
└─────────────────────────────────┘

[BƯỚC 1: Tín hiệu vật lý]       Chân RX (Pin) ──(0V / 3.3V)──> [Shift Register (UART)]
                                                                       │
                                                        (Nhận đủ 8-bit, copy byte)
                                                                       ▼
[BƯỚC 2: Ngoại vi UART]                                  [Data Register RDR]
                                                                       │
                                                       (Tự động bật cờ RXNE = 1)
                                                                       ▼
[BƯỚC 3: Kích hoạt Ngắt]                                 [Status Register SR]
                                                                       │
                                                      (Bật tín hiệu ngắt phần cứng)
                                                                       ▼
[BƯỚC 4: Bộ điều khiển Ngắt]                             [NVIC Controller]
                                                                       │
                              ┌────────────────────────────────────────┴────────────────────────────────────────┐
                              │ (Dừng luồng Main, Push R0-R3, LR, PC vào Stack)                                 │
                              ▼                                                                                 ▼
[BƯỚC 5: Lõi CPU Execution]  [Core Reg: PC] <── (Nạp địa chỉ ISR) ─── [Vector Table]              [Core Reg: R0] <── Đọc byte từ [RDR]
                              │                                                                                 │
                              └────────────────────────────────────────┬────────────────────────────────────────┘
                                                                       │
                                                    (CPU ghi R0 vào mảng RAM)
                                                                       ▼
[BƯỚC 6: Bộ nhớ RAM]                                     [RAM: Ring Buffer] (head++)
                                                                       │
                                             (Xóa cờ ngắt, Pop Stack, khôi phục luồng Main)
                                                                       ▼
[BƯỚC 7: Luồng Main FSM]     [RAM: Ring Buffer] ──(Pop byte)──> [Core Reg: R0/R1] ──(Giải mã bẫy ngoặc)──> [Control Register GPIO/PWM]

[BƯỚC 1: Tín hiệu vật lý]       Chân RX (Pin) ──(0V / 3.3V)──> [Shift Register (UART)]
                                                                       │
                                                        (Nhận đủ 8-bit, copy byte)
                                                                       ▼
[BƯỚC 2: Ngoại vi UART]                                  [Data Register RDR]
                                                                       │
                                                       (Tự động bật cờ RXNE = 1)
                                                                       ▼
[BƯỚC 3: Kích hoạt Ngắt]                                 [Status Register SR]
                                                                       │
                                                      (Bật tín hiệu ngắt phần cứng)
                                                                       ▼
[BƯỚC 4: Bộ điều khiển Ngắt]                             [NVIC Controller]
                                                                       │
                              ┌────────────────────────────────────────┴────────────────────────────────────────┐
                              │ (Dừng luồng Main, Push R0-R3, LR, PC vào Stack)                                 │
                              ▼                                                                                 ▼
[BƯỚC 5: Lõi CPU Execution]  [Core Reg: PC] <── (Nạp địa chỉ ISR) ─── [Vector Table]              [Core Reg: R0] <── Đọc byte từ [RDR]
                              │                                                                                 │
                              └────────────────────────────────────────┬────────────────────────────────────────┘
                                                                       │
                                                    (CPU ghi R0 vào mảng RAM)
                                                                       ▼
[BƯỚC 6: Bộ nhớ RAM]                                     [RAM: Ring Buffer] (head++)
                                                                       │
                                             (Xóa cờ ngắt, Pop Stack, khôi phục luồng Main)
                                                                       ▼
[BƯỚC 7: Luồng Main FSM]     [RAM: Ring Buffer] ──(Pop byte)──> [Core Reg: R0/R1] ──(Giải mã bẫy ngoặc)──> [Control Register GPIO/PWM]

BẢN CHẤT CẢM BIẾN / ĐỘNG CƠ                  KHỐI NGOẠI VI BÊN TRONG CHIP
 ┌─────────────────────────┐                 ┌─────────────────────────────┐
 │ Cảm biến dò Line IR     │──(Điện áp Vout)─>│ Mạch ADC (Tụ & Comparator)   │ ──> Thanh ghi ADC_DR
 ├─────────────────────────┤                 ├─────────────────────────────┤
 │ Mạch công suất Cầu H    │<──(Sóng PWM)────│ Mạch Timer (Flip-Flop Count)│ <── Thanh ghi TIM_CCR
 ├─────────────────────────┤                 ├─────────────────────────────┤
 │ Module UART / Bluetooth │<──(Xung TX/RX)──│ Shift Register (Bộ dịch bit)│ <── Thanh ghi USART_DR
 └─────────────────────────┘                 └─────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│                          VI ĐIỀU KHIỂN (MCU CHIP)                       │
│                                                                         │
│  ┌─────────────────┐       ┌─────────────────┐       ┌───────────────┐  │
│  │    LÕI CPU      │       │     BỘ NHỚ      │       │  KHỐI XUNG    │  │
│  │ (ALU, Core Reg) │       │ (Flash & SRAM)  │       │ (Clock/RCC)   │  │
│  └────────┬────────┘       └────────┬────────┘       └───────┬───────┘  │
│           │                         │                        │          │
│ ══════════╧═════════════════════════╧════════════════════════╧═════════ │ SYSTEM BUS
│           │                         │                        │          │
│  ┌────────┴────────┐       ┌────────┴────────┐       ┌───────┴───────┐  │
│  │   KHỐI TIMER    │       │   KHỐI UART     │       │   KHỐI ADC    │  │
│  │   (Mạch Đếm)    │       │  (Shift Reg)    │       │ (Sample-Hold) │  │
│  └────────┬────────┘       └────────┬────────┘       └───────┬───────┘  │
│           │                         │                        │          │
│  ┌────────┴─────────────────────────┴────────────────────────┴───────┐  │
│  │                  KHỐI I/O DRIVER (GPIO PINS)                      │  │
│  └──────────────────────────────────┬────────────────────────────────┘  │
└─────────────────────────────────────┼───────────────────────────────────┘
                                      │ (Chân cắm vật lý)
                                      ▼
                        THẾ GIỚI NGOẠI VI BÊN NGOÀI

┌─────────────────────────────────────────────────────────────────────────────┐
 │                         1. TẦNG CẢM BIẾN (SENSORS)                          │
 │      Cảm biến IR, Cảm biến Siêu âm, Encoder, Cảm biến Gia tốc IMU          │
 └──────────────────────────────────────┬──────────────────────────────────────┘
                                        │ (Tín hiệu Điện áp Analog / Digital)
                                        ▼
 ┌─────────────────────────────────────────────────────────────────────────────┐
 │               2. TẦNG ĐIỀU KHIỂN TRUNG TÂM (MICROCONTROLLER CORE)           │
 │   Chuyển đổi ADC/GPIO  ──>  Thanh ghi & RAM  ──>  Thuật toán FSM / PID      │
 └──────────────────────────────────────┬──────────────────────────────────────┘
                                        │ (Tín hiệu Điều khiển Logic 3.3V / PWM)
                                        ▼
 ┌─────────────────────────────────────────────────────────────────────────────┐
 │                  3. TẦNG MẠCH DRIVER (POWER AMPLIFIER LAYER)                │
 │   Mạch Cầu H (TB6612/L298), Transistor MOSFET, Optocoupler (Cách ly quang)   │
 └──────────────────────────────────────┬──────────────────────────────────────┘
                                        │ (Dòng điện công suất lớn 12V / 3A)
                                        ▼
 ┌─────────────────────────────────────────────────────────────────────────────┐
 │                    4. TẦNG CƠ CẤU CHẤP HÀNH (ACTUATORS)                     │
 │          Động cơ DC Bánh xe, Động cơ Servo Lái, Còi Buzzer, Đèn LED           │
 └─────────────────────────────────────────────────────────────────────────────┘